# 🐜 Optimisation par Colonie de Fourmis (Ant Colony Optimization – ACO)
### Résolution Interactive du Problème du Voyageur de Commerce (TSP)
**Auteur :** El Kachabi Badr-eddine  
**Profil GitHub :** [https://github.com/el-kachabi](https://github.com/el-kachabi)  
**Dépôt Officiel :** [https://github.com/el-kachabi/ant_colony_optimization](https://github.com/el-kachabi/ant_colony_optimization)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/el-kachabi/ant_colony_optimization/blob/main/notebooks/ant_colony_optimization.ipynb)


## 1. Installation des Dépendances & Environnement
Ce notebook est 100% autonome et peut s'exécuter directement dans Google Colab.

In [ ]:
# Installation des packages nécessaires pour Google Colab
!pip install -q numpy scipy pandas matplotlib plotly networkx

## 2. Modélisation Mathématique du TSP
Nous définissons ici la structure d'instance TSP avec calcul vectorisé de la matrice des distances $D$ et de la visibilité heuristique $\eta_{ij} = 1 / d_{ij}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import time
import itertools
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict

@dataclass(frozen=True)
class City:
    id: int
    name: str
    x: float
    y: float

@dataclass
class Tour:
    path: List[int]
    length: float
    is_valid: bool = True

class TSPInstance:
    def __init__(self, name: str, coords: List[Tuple[float, float]], labels: Optional[List[str]] = None, optimal_known: Optional[float] = None):
        self.name = name
        self.dimension = len(coords)
        self.optimal_known = optimal_known
        self.cities = [City(i, labels[i] if labels else f'C_{i}', float(c[0]), float(c[1])) for i, c in enumerate(coords)]
        
        # Matrice vectorisée des distances euclidiennes
        arr = np.array(coords, dtype=np.float64)
        diff = arr[:, np.newaxis, :] - arr[np.newaxis, :, :]
        self.distance_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
        
        # Matrice de visibilité heuristique eta = 1 / d (diagonale = 0)
        with np.errstate(divide='ignore', invalid='ignore'):
            self.heuristic_matrix = np.where(self.distance_matrix > 0, 1.0 / self.distance_matrix, 0.0)
        np.fill_diagonal(self.heuristic_matrix, 0.0)

    def compute_tour_length(self, path: List[int]) -> float:
        n = len(path)
        return sum(float(self.distance_matrix[path[i], path[(i + 1) % n]]) for i in range(n))

    def create_tour(self, path: List[int]) -> Tour:
        return Tour(path=list(path), length=self.compute_tour_length(path), is_valid=len(set(path)) == self.dimension)

# Création de l'instance didactique à 5 villes (optimum exact : 264.8528)
coords_5 = [(20.0, 20.0), (20.0, 80.0), (80.0, 80.0), (80.0, 20.0), (50.0, 50.0)]
inst_5 = TSPInstance('Micro_5_Villes', coords_5, labels=['V0', 'V1', 'V2', 'V3', 'V4'], optimal_known=264.8528)
print(f'Instance créée : {inst_5.name} ({inst_5.dimension} villes)')
print('Matrice des distances :\n', np.round(inst_5.distance_matrix, 2))

## 3. Moteur d'Optimisation Ant System (AS)
Implémentation des équations de Dorigo (1996) :
- Visibilité heuristique : $\eta_{ij} = 1 / d_{ij}$
- Règle de transition proportionnelle : $P_{ij}^k = \frac{[\tau_{ij}]^\alpha [\eta_{ij}]^\beta}{\sum [\tau_{il}]^\alpha [\eta_{il}]^\beta}$
- Évaporation : $\tau_{ij} \leftarrow (1 - \rho) \tau_{ij}$
- Dépôt par les fourmis : $\Delta \tau_{ij} = \sum Q / L_k$

In [ ]:
class AntSystem:
    def __init__(self, alpha=1.0, beta=3.0, rho=0.4, q=100.0, num_ants=20, max_iter=40, seed=42):
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q = q
        self.num_ants = num_ants
        self.max_iter = max_iter
        self.rng = np.random.default_rng(seed)

    def solve(self, instance: TSPInstance):
        n = instance.dimension
        tau0 = 1.0
        pheromone = np.full((n, n), tau0, dtype=np.float64)
        np.fill_diagonal(pheromone, 0.0)
        heuristic = instance.heuristic_matrix

        best_tour = None
        history = []

        for it in range(1, self.max_iter + 1):
            tours = []
            for k in range(self.num_ants):
                curr = int(self.rng.integers(0, n))
                path = [curr]
                unvisited = set(range(n))
                unvisited.remove(curr)

                while unvisited:
                    unv_list = list(unvisited)
                    weights = (pheromone[curr, unv_list] ** self.alpha) * (heuristic[curr, unv_list] ** self.beta)
                    s_w = np.sum(weights)
                    probs = weights / s_w if s_w > 0 else np.full(len(unv_list), 1.0 / len(unv_list))
                    next_c = unv_list[int(self.rng.choice(len(unv_list), p=probs))]
                    path.append(next_c)
                    unvisited.remove(next_c)
                    curr = next_c

                tour = instance.create_tour(path)
                tours.append(tour)

            iter_best = min(tours, key=lambda t: t.length)
            if best_tour is None or iter_best.length < best_tour.length:
                best_tour = iter_best

            # Évaporation et Dépôt
            pheromone = (1.0 - self.rho) * pheromone
            delta = np.zeros((n, n), dtype=np.float64)
            for t in tours:
                dep = self.q / t.length
                for i in range(n):
                    u, v = t.path[i], t.path[(i + 1) % n]
                    delta[u, v] += dep
                    delta[v, u] += dep
            pheromone += delta
            np.fill_diagonal(pheromone, 0.0)
            pheromone = np.maximum(pheromone, 1e-6)

            history.append({'iter': it, 'best_len': best_tour.length, 'mean_len': np.mean([t.length for t in tours])})

        return best_tour, history, pheromone

# Test sur 5 villes
aco = AntSystem(alpha=1.0, beta=3.0, rho=0.4, num_ants=15, max_iter=30, seed=42)
best_tour, history, phero_mat = aco.solve(inst_5)
print(f'Meilleure tournée trouvée : {best_tour.path} -> {best_tour.path[0]}')
print(f'Longueur obtenue : {best_tour.length:.4f} (Optimum exact : {inst_5.optimal_known:.4f})')
print(f'Écart à l\'optimum : {abs(best_tour.length - inst_5.optimal_known) / inst_5.optimal_known * 100:.2f}%')

## 4. Visualisation Graphique de la Convergence et de la Tournée

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Tracé de la tournée
xs = [inst_5.cities[i].x for i in best_tour.path] + [inst_5.cities[best_tour.path[0]].x]
ys = [inst_5.cities[i].y for i in best_tour.path] + [inst_5.cities[best_tour.path[0]].y]
ax1.plot(xs, ys, 'o-', color='#00F2FE', linewidth=2.5, markersize=8, label=f'Tournée (L = {best_tour.length:.2f})')
for c in inst_5.cities:
    ax1.annotate(c.name, (c.x, c.y), textcoords='offset points', xytext=(0, 10), ha='center', fontweight='bold')
ax1.set_title(f'Tournée Optimale ({inst_5.name})', fontsize=12, fontweight='bold')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

# 2. Courbe de convergence
iters = [h['iter'] for h in history]
bests = [h['best_len'] for h in history]
means = [h['mean_len'] for h in history]
ax2.plot(iters, bests, 'o-', color='#00F2FE', linewidth=2, label='Meilleure Distance')
ax2.plot(iters, means, '--', color='#FF9900', linewidth=1.5, label='Distance Moyenne')
ax2.axhline(inst_5.optimal_known, color='#10B981', linestyle=':', label=f'Optimum Référence ({inst_5.optimal_known:.2f})')
ax2.set_title('Dynamique de Convergence Multi-Générations', fontsize=12, fontweight='bold')
ax2.set_xlabel('Itérations')
ax2.set_ylabel('Longueur')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Tournoi Comparatif : ACO vs Baselines (Glouton, 2-Opt, Brute Force)
Comparons les performances de l'ACO face aux méthodes classiques de recherche opérationnelle.

In [ ]:
# Solveur Glouton (Plus Proche Voisin)
def solve_nearest_neighbor(instance: TSPInstance):
    n = instance.dimension
    visited = [0]
    unvisited = set(range(1, n))
    curr = 0
    while unvisited:
        nxt = min(unvisited, key=lambda c: instance.distance_matrix[curr, c])
        visited.append(nxt)
        unvisited.remove(nxt)
        curr = nxt
    return instance.create_tour(visited)

# Solveur Exact Brute Force (pour N <= 10)
def solve_brute_force(instance: TSPInstance):
    n = instance.dimension
    best_t = None
    for perm in itertools.permutations(range(1, n)):
        if perm[0] > perm[-1]: continue  # Élimine la symétrie miroir
        t = instance.create_tour([0] + list(perm))
        if best_t is None or t.length < best_t.length:
            best_t = t
    return best_t

nn_tour = solve_nearest_neighbor(inst_5)
exact_tour = solve_brute_force(inst_5)

summary = pd.DataFrame([
    {'Méthode': 'Exact Brute Force', 'Longueur': exact_tour.length, 'Écart Optimum': '0.00%'},
    {'Méthode': 'Ant System (AS)', 'Longueur': best_tour.length, 'Écart Optimum': f'{(best_tour.length - exact_tour.length)/exact_tour.length*100:+.2f}%'},
    {'Méthode': 'Nearest Neighbor (Glouton)', 'Longueur': nn_tour.length, 'Écart Optimum': f'{(nn_tour.length - exact_tour.length)/exact_tour.length*100:+.2f}%'},
])
print(summary.to_string(index=False))

## 6. Conclusion et Perspectives
Ce notebook démontre que l'algorithme d'optimisation par colonie de fourmis (ACO) est capable d'atteindre l'optimum mathématique global ($264.85$) en combinant harmonieusement la mémoire collective des traces de phéromones et la visibilité heuristique.

Pour explorer la simulation microscopique interactive et l'analyse de sensibilité 2D, lancez l'application Streamlit du projet :
```bash
streamlit run app.py
```

**Auteur :** El Kachabi Badr-eddine ([GitHub @el-kachabi](https://github.com/el-kachabi))